# HAILA: ajuste fino do Qwen com LoRA

Antes de começar: **Ambiente de execução > Alterar o tipo de ambiente > GPU T4**.

Rode as células em ordem. Você vai precisar enviar dois arquivos: o zip do projeto (`HAILACBQs-corrigido.zip`) e o `questoes.jsonl` gerado pelo `extrair_enade.py`.

In [ ]:
# 1. Confere a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Envie o zip do projeto e o questoes.jsonl
from google.colab import files
enviados = files.upload()
print(list(enviados))

In [ ]:
# 3. Descompacta e instala dependências
!unzip -q -o HAILACBQs-corrigido.zip
%cd HAILACBQs-main
!pip install -q "transformers>=4.44" "peft>=0.12" "accelerate>=0.33"

In [ ]:
# 4. Prepara os dados (filtros + divisão treino/validação)
!python research/lora/preparar_dados.py ../questoes.jsonl dados/haila-lora-v1

In [ ]:
# 5. Treina o LoRA (leva de alguns minutos a ~1 h, conforme o número de questões)
!python research/lora/treinar_lora_qwen.py dados/haila-lora-v1 adapters/qwen-distratores-lora-v1

In [ ]:
# 6. Mostra o resultado na validação
import json
cfg = json.load(open('adapters/qwen-distratores-lora-v1/treino_config.json'))
print(json.dumps(cfg['resultado'], ensure_ascii=False, indent=2))

In [ ]:
# 7. Baixa o adapter e os relatórios de dados
!rm -rf adapters/qwen-distratores-lora-v1/checkpoints
!cp dados/haila-lora-v1/relatorio_dados.json adapters/qwen-distratores-lora-v1/
!cd adapters && zip -qr ../qwen-distratores-lora-v1.zip qwen-distratores-lora-v1
files.download('qwen-distratores-lora-v1.zip')

Descompacte `qwen-distratores-lora-v1.zip` dentro da pasta `adapters/` do projeto na sua máquina e configure no `.env`:

```
HAILA_SLM_ADAPTER_PATH=adapters/qwen-distratores-lora-v1
```

Depois confira em `http://127.0.0.1:8000/health` que `slm_mode` é `lora`.